# PHASE 2 — REGRESSION


# Day 09 — Regression Evaluation


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Calculate and interpret MAE, MSE, RMSE, and $R^2$.
- Explain the mathematical difference between these metrics.
- Understand when MAE is better than RMSE (and vice versa) based on outliers.
- Understand the concept of Adjusted $R^2$ for multiple regression.


## 2. Prerequisites
- Day 8 (Linear Regression).


## 3. Concept
In regression, our predictions are almost never 100% perfectly accurate. There is always an error (or residual): $e_i = y_i - \hat{y}_i$.

To evaluate a model, we aggregate these errors into a single score. However, *how* we aggregate them drastically changes how the model is penalized for being wrong.

- **MAE** (Mean Absolute Error): The average absolute distance from the truth.
- **MSE** (Mean Squared Error): The average squared distance.
- **RMSE** (Root Mean Squared Error): The square root of MSE, bringing it back to the original units.
- **$R^2$**: The proportion of variance in the target explained by the model.


## 4. Why Does This Matter?
If you are predicting House Prices, being off by $10,000 on a $500,000 house is not a big deal. Being off by $500,000 on a $500,000 house is disastrous. 
Different metrics punish large errors (outliers) differently. Choosing the wrong metric leads to selecting the wrong model for your business problem.


## 5. Intuition
- **MAE**: "On average, my house price predictions are off by $15,000."
- **RMSE**: "On average, my predictions are off by roughly $18,000, but I am penalizing huge misses more heavily."
- **$R^2$**: "My model explains 85% of the reasons why some houses are more expensive than others."


## 6. Mathematical Foundation
**MAE** = $\frac{1}{n} \sum |y_i - \hat{y}_i|$

**MSE** = $\frac{1}{n} \sum (y_i - \hat{y}_i)^2$

**RMSE** = $\sqrt{MSE}$

**$R^2$** = $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$ (where $\bar{y}$ is the mean of the true targets).

Notice that MSE and RMSE *square* the error. If an error is $10$, squaring it makes it $100$. This means RMSE violently punishes the model if it makes a few massive mistakes, whereas MAE treats all mistakes linearly.


## 7. Scikit-learn API
Scikit-learn provides these functions inside the `metrics` module:

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False) # or np.sqrt(mse)
r2 = r2_score(y_true, y_pred)
```


## 8. Simple Example
Let's generate data with ONE massive outlier and see how it affects MAE vs RMSE.


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# True values (House Prices in thousands)
y_true = np.array([100, 100, 100, 100, 1000]) # 1000 is a massive mansion

# Model A: Predicts perfectly on normal houses, but completely misses the mansion
y_pred_A = np.array([100, 100, 100, 100, 0])

# Model B: Mediocre predictions on everything
y_pred_B = np.array([300, 300, 300, 300, 700])

print('Model A:')
print(f'MAE:  {mean_absolute_error(y_true, y_pred_A):.2f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_true, y_pred_A)):.2f}')

print('\nModel B:')
print(f'MAE:  {mean_absolute_error(y_true, y_pred_B):.2f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_true, y_pred_B)):.2f}')


## 9. Code Walkthrough
- **Model A** has a much lower MAE (200 vs 220). So according to MAE, Model A is better.
- **Model B** has a much lower RMSE (223 vs 447). So according to RMSE, Model B is better.

Why? Because Model A missed the mansion by $1,000! RMSE squared that $1,000 to $1,000,000, violently punishing Model A. Model B missed the normal houses by $200 and the mansion by $300, but RMSE didn't care as much because none of those misses were as massively extreme as $1,000.


## 10. Experiment
When is MAE better than RMSE?
If your dataset has bizarre outliers (like a data entry error where someone typed $15,000,000 instead of $150,000), RMSE will force your model to bend towards the error to avoid the massive squared penalty. 

**Use MAE** when you want to ignore outliers.
**Use RMSE** when large errors are completely unacceptable to your business.


In [ ]:
# Example of $R^2$
print(f'Model A R^2: {r2_score(y_true, y_pred_A):.4f}')
print(f'Model B R^2: {r2_score(y_true, y_pred_B):.4f}')


## 11. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
y_actual = np.array([10, 20, 30, 40])
y_predict = np.array([10, 20, 30, 40])
perfect_r2 = r2_score(y_actual, y_predict)

y_mean_predict = np.array([25, 25, 25, 25])
mean_r2 = r2_score(y_actual, y_mean_predict)


> **Question:** What is the exact value of `perfect_r2`? What is the exact value of `mean_r2`?

**Think before running the next cell!**


In [ ]:
print('Perfect R^2:', perfect_r2)
print('Mean Prediction R^2:', mean_r2)
print('\nWhy? A perfect model explains 100% (1.0) of the variance. A model that just predicts the average explains 0% (0.0) of the variance.')


## 12. Coding Exercise
Write a function `evaluate_regression(y_true, y_pred)` that prints the MAE, MSE, RMSE, and $R^2$ neatly formatted to 2 decimal places.


In [ ]:
# YOUR CODE HERE
def evaluate_regression(y_t, y_p):
    mae = mean_absolute_error(y_t, y_p)
    mse = mean_squared_error(y_t, y_p)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_t, y_p)
    print(f'MAE:  {mae:.2f}')
    print(f'MSE:  {mse:.2f}')
    print(f'RMSE: {rmse:.2f}')
    print(f'R^2:  {r2:.2f}')

# Test it
evaluate_regression(y_true, y_pred_A)


## 13. Debugging Challenge
The junior data scientist tried to evaluate their model, but got a wildly negative $R^2$ score. Find the bug!


In [ ]:
# Buggy code
try:
    y_true_bug = np.array([1, 2, 3, 4, 5])
    y_pred_bug = np.array([1.1, 2.1, 2.9, 4.2, 5.0])
    
    # THE BUG IS HERE:
    bad_r2 = r2_score(y_pred_bug, y_true_bug)
    print('R^2:', bad_r2)
except Exception as e:
    print('Error:', e)


> **Hint:** Scikit-learn metrics ALWAYS expect the true labels first, then the predictions: `metric(y_true, y_pred)`. Reversing them calculates the variance of the *predictions*, not the truth, which ruins the math!


## 14. Model Evaluation (Adjusted R-Squared)
$R^2$ has a fatal flaw: if you add 1,000 completely random, useless features to your dataset, $R^2$ will technically go up (or stay flat) simply due to random chance correlations. It never goes down.

**Adjusted $R^2$** penalizes you for adding useless features:

$$ R^2_{adj} = 1 - \left( \frac{(1 - R^2)(n - 1)}{n - p - 1} \right) $$
Where $n$ is sample size and $p$ is the number of features. Scikit-learn doesn't have a built-in function for this, but it's easy to calculate manually!


## 15. Real-World Example
In ride-sharing apps (like Uber/Lyft), predicting ETA is a regression task. 
If the RMSE is huge, it means occasionally the app predicts a 5-minute wait, but the driver arrives in 30 minutes. This ruins user trust! Therefore, ride-sharing companies optimize heavily for RMSE to avoid massive edge-case failures, rather than just MAE.


## 16. Mini Project
Calculate the Adjusted $R^2$ for Model A. Assume the dataset had 5 samples ($n=5$) and 2 features ($p=2$).


In [ ]:
n = len(y_true)
p = 2
r2_A = r2_score(y_true, y_pred_A)

adj_r2 = 1 - ( (1 - r2_A) * (n - 1) / (n - p - 1) )
print(f'R^2: {r2_A:.4f}')
print(f'Adjusted R^2: {adj_r2:.4f}')
print('Notice how Adjusted R^2 is lower, heavily penalizing the model because we used 2 features for only 5 rows of data!')


## 17. Common Mistakes
- **Comparing MAE to RMSE directly**: MAE will almost always be lower than RMSE. You cannot compare them to each other. You must compare MAE of Model A to MAE of Model B.
- **Passing strings to regression metrics**: Regression metrics require continuous numbers. You cannot calculate MSE on `['cat', 'dog']`.
- **Swapping `y_true` and `y_pred`**: Always `y_true` first.


## 18. Interview Questions
- **Beginner**: What does an $R^2$ of 0 mean?
- **Intermediate**: Why might RMSE be a better metric than MAE when predicting the dosage of a dangerous medication?
- **Advanced**: Why does $R^2$ naturally increase as you add more features, and how does Adjusted $R^2$ fix this?


## 19. Knowledge Check
- Which metric squares the errors before averaging? (MSE / RMSE)
- Which metric treats all errors linearly? (MAE)


## 20. Summary
- **MAE**: Easy to interpret. Resilient to outliers.
- **RMSE**: Punishes large errors. Sensitive to outliers.
- **$R^2$**: Percentage of variance explained.
- **Adjusted $R^2$**: Fixes $R^2$'s vulnerability to useless features.


## 21. Homework
Write a loop that calculates RMSE and MAE for an array. Then, inject a massive outlier into the array (e.g., change one prediction to be wrong by 1,000,000) and observe how RMSE explodes compared to MAE.
